# nitid Tutorial 2: Train and Evaluate a Model
This notebook walks through the nitid workflow for training and evaluating models:

1. Set up the project environment for training
2. Download a dataset
3. Train a model
4. Evaluate and inspect the results
5. Export the model


This notebook runs locally from a nitid checkout.

# Set up the project environment for training

Install the training extra from the repository root:
```bash
uv sync --extra train
```

The cell below finds the repository root and works inside `tutorials/`.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if all((candidate / p).exists() for p in ["dfine", "tools", "configs"]):
            return candidate
    raise RuntimeError(
        "Could not find the nitid repository root. "
        "Run this notebook from the cloned nitid repo, or `cd` into it first."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
TUTORIAL_DIR = REPO_ROOT / "tutorials"
os.chdir(REPO_ROOT)

print(f"Running in Colab: {IN_COLAB}")
print(f"Repository root: {REPO_ROOT}")
print(f"Tutorial directory: {TUTORIAL_DIR}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
from nitid import NITID

print("Imported NITID successfully")

# Download a Dataset

nitid accepts either COCO JSON annotations or YOLO .txt labels for detection and instance segmentation.

Semantic segmentation instead uses one dense class-ID PNG mask per image.

**COCO JSON**

Your dataset directory can look like:
```
my_dataset/
  images/
    train/
      000001.jpg
      ...
    val/
      000100.jpg
      ...
  annotations/
    instances_train.json
    instances_val.json
```

Annotation files follow the standard COCO detection format. For task="segment", every object must also contain a COCO polygon or RLE segmentation field.


**YOLO .txt**

Standard Ultralytics-style layout is also supported:

```
my_dataset/
  images/
    train/
    val/
  labels/
    train/
    val/
```
Split-first layouts are supported too:

```
my_dataset/
  train/
    images/
    labels/
  val/
    images/
    labels/
```

nitid detects the layout automatically and converts YOLO labels to cached COCO JSON internally for training and validation. The generated cache is stored in nitid's user cache directory rather than inside the dataset tree.

Detection rows use the usual box representation:

`class x_center y_center width height`

Instance-segmentation rows use normalized polygon points:

`class x1 y1 x2 y2 x3 y3 ...`

Bounding-box-only annotations are rejected when training a segmentation model, preventing an accidental all-zero mask training run.


**Dense semantic masks**

For task="semantic", image and mask paths mirror one another:

```
my_semantic_dataset/
  images/
    train/example.jpg
    val/example.jpg
  labels/
    train/example.png
    val/example.png
```

Each PNG must be a single-channel integer class map with the same dimensions as its image. Pixel values are contiguous class IDs from 0 through nc - 1. The configured ignore_index (255 by default) marks pixels excluded from loss and validation metrics. Dense masks are always resized with nearest-neighbor sampling; padding introduced by geometric augmentation is filled with the ignore index.

```
path: /data/my_semantic_dataset
train: images/train
val: images/val
nc: 3
names:
  0: background
  1: road
  2: vehicle
ignore_index: 255
```

When image and mask directories do not follow the mirrored images/labels layout, set train_masks: and val_masks: explicitly. Semantic training does not accept instance-only mosaic, mixup, classes, or single_cls options.

**Data YAML** 
Point to your dataset with an ultralytics-style YAML:

```
# configs/datasets/my_dataset.yml
path: /data/my_dataset
train: images/train
val:   images/val

nc: 3
names:
  0: person
  1: car
  2: bicycle
```

When model.train(...) starts, nitid reads this taxonomy and automatically rebuilds the class head when it differs from the checkpoint. Backbone and localization weights are retained, the pretrained encoder scorer is converted to generic objectness for proposal selection, and only taxonomy-specific class heads are initialized from scratch. No manual head replacement is required.

nitid derives annotation file paths automatically: <path>/annotations/instances_<split_dirname>.json

Override with explicit keys if your layout differs:

```
train_ann: annotations/my_train.json
val_ann:   annotations/my_val.json
For YOLO datasets, use the same YAML shape and point train: / val: at the image directories:

path: /data/my_dataset
train: images/train
val:   images/val

names:
  0: person
  1: car
  2: bicycle
```

or, for split-first layouts:
```
path: /data/my_dataset
train: train/images
val:   val/images

names:
  0: person
  1: car
  2: bicycle
```

**Category ID mapping**
COCO annotation files use arbitrary category_id integers (often starting at 1, or non-contiguous). nitid needs 0-based label indices for the model head. By default it sorts the category_id values found in the annotation file and maps them to 0, 1, 2, … in that order.

If your IDs are non-contiguous or in a custom order, supply an explicit mapping:

```
# annotation file has category_id 1 and 3 (no 2)
cat_ids:
  1: 0   # category_id 1  →  label index 0  (person)
  3: 1   # category_id 3  →  label index 1  (car)
```

Without this override, the auto-mapping would produce the same result for IDs {1, 3} because they sort to [1, 3] → [0, 1]. The override is only needed when you want a different ordering than sorted order.

In [ ]:
%pip install roboflow

In [ ]:
# Download and unzip a dataset from Roboflow.
# Set ROBOFLOW_API_KEY in your environment, or paste your key when prompted.
from getpass import getpass

from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("school-bgh12").project("smart-street-lighting")
version = project.version(7)
dataset = version.download("coco")

In [ ]:
# This step is optional
import shutil

src = REPO_ROOT / "Smart-Street-Lighting-7"  # change to your dataset folder name
dst = TUTORIAL_DIR / src.name

shutil.move(str(src), str(dst))
print(f"Moved to: {dst}")

Verify that the folder format matches the following:

```
my_dataset/
  images/
    train/
      000001.jpg
      ...
    val/
      000100.jpg
      ...
  annotations/
    instances_train.json
    instances_val.json
```    
If it doesn't, we need to adjust it.

# Train a model
The default recipe follows the task-native training path. For detection this
uses D-FINE-style defaults, matching, losses, and augmentations.

This example trains a detection model from `model1s`. You can replace `model1s`
with `model1m`, `model1l`, or `model1x`.

* For segmentation models: `model = NITID("model1s", task="segment")`
* For semantic models: `model = NITID("model1s", task="semantic")`


In [ ]:
DATA_YAML = TUTORIAL_DIR / "Smart-Street-Lighting-7/data.yaml"  # path to the dataset YAML

model = NITID("model1s", task="detect")  # or "model1m"

metrics = model.train(
    data=DATA_YAML,
    epochs=20,  # short example run; around 200 epochs is recommended
    imgsz=640,
    batch=16,  # 32 is recommended if it fits in memory
    save_period=-1,  # save only best.pth and last.pth; set N to also save every N epochs
    project=TUTORIAL_DIR / "runs/train",
    name="my_experiment",
)

print(metrics)

# Evaluate and inspect the results
1) Display the validation plots of the trained model (confusion matrix, curves, etc.)
2) Display predictions on the dataset's test images

In [ ]:
metrics = model.val(
    data=DATA_YAML,
    project=TUTORIAL_DIR / "runs/val",
    name="exp",
)

In [ ]:
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    display = None

if Image is not None and display is not None:
    for imgs in os.listdir(TUTORIAL_DIR / "runs/val/exp"):
        if ".png" in imgs:
            result_path = TUTORIAL_DIR / "runs/val/exp" / imgs
            display(Image(filename=str(result_path)))

In [ ]:
TEST_PATH = TUTORIAL_DIR / "Smart-Street-Lighting-7/images/test"
SAVE_PATH = TUTORIAL_DIR / "outputs/test"
SAVE_PATH.mkdir(parents=True, exist_ok=True)


IMGZ = 640
CONF = 0.5

for imgs in os.listdir(TEST_PATH):
    results = model.predict(f"{TEST_PATH}/{imgs}", conf=CONF, imgsz=IMGZ)

    if not results:
        print("No results returned for:", imgs)
        continue

    result = results[0]
    detections = result.to_json()
    print(result)
    print("Detections:", len(result))

    annotated_path = SAVE_PATH / imgs
    # results.save() writes an annotated image with boxes and labels.
    result.save(str(annotated_path))
    print(f"Saved annotated image to: {annotated_path}")

# Export the model
Export the trained model to ONNX, OpenVINO and TorchScript. Exported graphs include the postprocessor, so they return `(labels, boxes, scores)`. See the [export guide](../docs/export.md) for TensorRT and the other options.

ONNX export:

In [ ]:
onnx_path = model.export(format="onnx", output=TUTORIAL_DIR / "runs/export/model.onnx")
# CLI: nitid export model=path/to/best.pth format=onnx output=runs/export/model.onnx

OpenVINO export needs the optional OpenVINO dependencies:
```bash
uv sync --extra openvino
```

In [ ]:
xml_path = model.export(format="openvino", output=TUTORIAL_DIR / "runs/export/model.xml")
# CLI: nitid export model=path/to/best.pth format=openvino output=runs/export/model.xml

In [ ]:
# Load the exported model with OpenVINO Runtime
import openvino as ov

core = ov.Core()
compiled_model = core.compile_model(xml_path, "AUTO")
print("model compiled")

TorchScript export:

In [ ]:
model.export(format="torchscript", output=TUTORIAL_DIR / "runs/export/model.torchscript")
# CLI: nitid export model=path/to/best.pth format=torchscript output=runs/export/model.torchscript